# Modelling Training

In [ ]:
%load_ext autoreload
%autoreload 2
from postgresql import get_engine
from datetime import datetime as dt
import pandas as pd
pd.set_option("display.max_columns", 100)
pd.set_option('display.max_rows', 50)


In [ ]:
engine = get_engine()

In [ ]:
import os
from src.config import ROOT_DIR

input_dir = f'{ROOT_DIR}/data/processed/'
input_file_path = os.path.join(input_dir, "merged_features_data.csv")

In [ ]:
df_features = pd.read_csv(input_file_path)
df_features

In [ ]:
df_pro_matches = pd.read_sql(
    'pro_matches',
    engine,
)

df_pro_matches

In [ ]:
# Extract start_time and match_outcome from pro_matches and concatenate to df
df = pd.merge(
    df_pro_matches[['match_id', 'start_time', 'radiant_win']],
    df_features,
    on='match_id',
    how='right'
)

In [ ]:
df

In [ ]:
# Convert string to datetime for start_time
df['start_time'] = pd.to_datetime(df['start_time'], unit='s')

In [ ]:
train = df.loc[df['start_time'] < dt(2024,6,6)]
valid = df.loc[(df['start_time'] > dt(2024,6,6)) & (df['start_time'] < dt(2024,8,6))]
test = df.loc[df['start_time'] > dt(2024,8,6)]

In [ ]:
len(test)/len(train)

In [ ]:
len(valid)/len(train)

In [ ]:
# Performing necessary transformations
def process_label(df):
    label = df['radiant_win']
    label = label.apply(int)
    return label.values

def process_features(df):
    features = df.drop(columns=['start_time','radiant_win','match_id'])
    features = features.dropna()
    return features.values
    


In [ ]:
y_train, y_valid, y_test = process_label(train), process_label(valid), process_label(test)
X_train, X_valid, X_test = process_features(train), process_features(valid), process_features(test)

In [ ]:
from collections import Counter

print(Counter(y_train), Counter(y_valid) , Counter(y_test))
# Dataset is not imbalanced 


In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
y_train.shape

In [ ]:
y_test.shape

In [ ]:
from sklearn.ensemble import RandomForestClassifier 

rf_clf = RandomForestClassifier()
rf_clf.fit(X_train, y_train)




In [ ]:
from joblib import dump, load

In [ ]:
# Persisting the model

output_dir = f"{ROOT_DIR}/models"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, 'random_forest_model.joblib')
# dump(rf_clf, output_path)


## Model Performance Validation

In [ ]:
clf = load(output_path)

In [ ]:
y_pred = clf.predict(X_valid)

In [ ]:
%matplotlib inline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
cm = confusion_matrix(y_valid, y_pred, labels=clf.classes_)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=clf.classes_)
disp.plot()
plt.show()


In [ ]:
from sklearn.metrics import accuracy_score

accuracy_score(y_valid, y_pred) 


## Persisting model with bentoML

In [ ]:
import bentoml


saved_model = bentoml.sklearn.save_model(
    name="rf_model",
    model=clf, 
    signatures={'predict':{'batchable':True}}
)

print(f"Model saved: {saved_model}")

In [ ]:
# Prediction with bentoml model service
import requests
import json


In [ ]:
output_list = X_test.tolist()
output_list

In [ ]:
# Wrap the features in another layer with "input_data" key
request_data = {"input_data": {"features": X_test.tolist()}}

response = requests.post(
    "http://localhost:3333/predict",
    headers={"Content-Type": "application/json"},
    data=json.dumps(request_data)
)

# Process the response
if response.status_code == 200:
    result = response.json()
    print("Prediction result:", result)
else:
    print(f"Error: {response.status_code}")
    print(response.text)